# Cantilever Beam — Deflection & Stress Sizing

**User story:** As a structural engineer preparing a submittal package for an EPC contractor,
I need to verify that a proposed steel support beam meets deflection and stress limits
so the contractor can confirm the member sizing before fabrication.

This notebook derives the expected tip deflection and maximum bending stress
for a rectangular cantilever beam under a point load at the free end,
then validates the FEM simulation against these closed-form results.

## Assumptions

- Homogeneous, isotropic, linear elastic material (ASTM A36 structural steel)
- Small deflections (Euler-Bernoulli beam theory valid)
- Rectangular prismatic cross-section (no taper, no holes)
- Load applied at centroid of free-end face (no eccentricity)
- Fixed end is perfectly clamped (no rotational compliance)
- Self-weight neglected (load >> beam weight for this geometry)
- No shear deformation (slender beam: L/h = 20 >> 5)
- Room temperature (no thermal effects on material properties)

In [1]:
import sympy as sp
import pint
from uncertainties import ufloat

ureg = pint.UnitRegistry()

# --- Symbols for derivation ---
P, L, E, I, b, h, x, c = sp.symbols('P L E I b h x c', positive=True)

## Derivation: Second Moment of Area

For a rectangular cross-section:

In [2]:
I_rect = b * h**3 / 12
sp.Eq(I, I_rect)

Eq(I, b*h**3/12)

## Derivation: Maximum Tip Deflection

Euler-Bernoulli beam equation, cantilever with point load at free end.
Integrate the moment-curvature relation with BCs: y(0)=0, y'(0)=0.

In [3]:
# Reason: moment at distance x from fixed end: M(x) = P*(L - x)
M = P * (L - x)

# Reason: curvature EI*y'' = M(x), integrate twice with BCs y(0)=0, y'(0)=0
y_double_prime = M / (E * I)
y_prime = sp.integrate(y_double_prime, x)  # slope
y = sp.integrate(y_prime, x)  # deflection

# Evaluate at x = L for max deflection
delta_max = sp.simplify(y.subs(x, L))
sp.Eq(sp.Symbol('delta_max'), delta_max)

Eq(delta_max, L**3*P/(3*E*I))

## Derivation: Maximum Bending Stress

Maximum moment is at the fixed end (x=0): M_max = P*L.
Maximum bending stress at the outer fiber (c = h/2).

In [4]:
M_max = P * L
sigma_max = M_max * c / I
sp.Eq(sp.Symbol('sigma_max'), sigma_max)

Eq(sigma_max, L*P*c/I)

## Numerical Evaluation

In [5]:
# --- Physical parameters with units and uncertainty ---
BEAM_LENGTH = 200.0 * ureg.mm
BEAM_WIDTH = 20.0 * ureg.mm
BEAM_HEIGHT = 10.0 * ureg.mm
YOUNGS_MODULUS = ufloat(200.0, 4.0) * ureg.GPa  # +/-2%, ASTM A36 typical
POINT_LOAD = 100.0 * ureg.N

# Second moment of area
I_val = (BEAM_WIDTH * BEAM_HEIGHT**3 / 12).to(ureg.mm**4)
print(f"I = {I_val}")

# Tip deflection: delta = P*L^3 / (3*E*I)
# Reason: working in mm/N/MPa system — convert E from GPa to MPa
p = POINT_LOAD.to(ureg.N).magnitude
l = BEAM_LENGTH.to(ureg.mm).magnitude
e = YOUNGS_MODULUS.magnitude.nominal_value * 1e3  # GPa -> MPa
i = I_val.magnitude

delta_val = p * l**3 / (3 * e * i)
print(f"Expected tip deflection: {delta_val:.4f} mm")

# Max bending stress: sigma = P*L*c / I
c_val = BEAM_HEIGHT.to(ureg.mm).magnitude / 2
sigma_val = p * l * c_val / i
print(f"Expected max bending stress: {sigma_val:.2f} MPa")

I = 1666.6666666666667 millimeter ** 4
Expected tip deflection: 0.8000 mm
Expected max bending stress: 60.00 MPa


## Expected Values

```
Tip deflection:      0.80 mm +/- 5% (Euler-Bernoulli vs FEM mesh stiffness)
Max bending stress:  60.0 MPa +/- 10% (stress concentration at fixed support)
```